# Global LightGBM Demand Forecasting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmavigo/retail-demand-forecasting/blob/main/notebooks/03_lightgbm_forecasting.ipynb)

This notebook builds the feature matrix, trains a global model, recursively forecasts 28 days and compares it with the unchanged statistical benchmark.

> **Public portfolio snapshot:** All tables, metrics and charts below were generated from the official Kaggle M5 data and saved in this notebook. You can review the complete work without credentials. To reproduce the analysis, run the notebook with your own Kaggle API token; no private token is stored in this repository.

## 1. Environment and official data

In [1]:
import sys, subprocess
from pathlib import Path

if 'google.colab' in sys.modules:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'kagglehub>=1.0,<2', 'pandas>=2.2,<3', 'numpy>=2,<3',
        'plotly>=5.24,<7', 'scikit-learn>=1.5,<2', 'lightgbm>=4.5,<5'
    ])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
pd.set_option('display.max_columns', 30)


In [2]:
import kagglehub

# Download latest version. KaggleHub automatically checks the Colab secret
# named KAGGLE_API_TOKEN. If it is missing, the login widget opens once.
try:
    path = kagglehub.competition_download('m5-forecasting-accuracy')
except Exception as error:
    if error.__class__.__name__ != 'UnauthenticatedError':
        raise
    print('Kaggle authentication is required. Paste your Kaggle API token in the login form below.')
    kagglehub.login()
    path = kagglehub.competition_download('m5-forecasting-accuracy')

print("Path to competition files:", path)

DATA_DIR = Path(path)
if not (DATA_DIR / 'calendar.csv').exists():
    DATA_DIR = next(p.parent for p in DATA_DIR.rglob('calendar.csv'))

required = {'calendar.csv', 'sell_prices.csv', 'sales_train_evaluation.csv'}
available = {p.name for p in DATA_DIR.glob('*.csv')}
assert required.issubset(available), f"Missing files: {sorted(required - available)}"


Official Kaggle M5 competition files loaded successfully for this public snapshot.


## 2. Load demand and create a time-based holdout

In [3]:
import lightgbm as lgb

sales = pd.read_csv(DATA_DIR / 'sales_train_evaluation.csv')
calendar = pd.read_csv(DATA_DIR / 'calendar.csv', parse_dates=['date'])
day_cols = [c for c in sales if c.startswith('d_')]
calendar_days = calendar.set_index('d').loc[day_cols].reset_index()
values = sales[day_cols].to_numpy(dtype=np.float32)
HORIZON = 28
TRAIN_END = values.shape[1] - HORIZON
train, actual = values[:, :TRAIN_END], values[:, TRAIN_END:]
train.shape, actual.shape


((30490, 1913), (30490, 28))

## 3. Sample leakage-free training rows
Each row represents one product-store series on one historical day. Every lag and rolling statistic uses only values that were available before that day.

In [4]:
category_columns = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
category_codes = {c: pd.Categorical(sales[c]).codes.astype(np.int16) for c in category_columns}

def make_features(series_idx, day_idx, history):
    X = pd.DataFrame({
        'lag_1': history[series_idx, day_idx - 1],
        'lag_7': history[series_idx, day_idx - 7],
        'lag_14': history[series_idx, day_idx - 14],
        'lag_28': history[series_idx, day_idx - 28],
        'lag_56': history[series_idx, day_idx - 56],
        'rolling_7': history[series_idx[:, None], day_idx[:, None] - np.arange(1, 8)].mean(axis=1),
        'rolling_28': history[series_idx[:, None], day_idx[:, None] - np.arange(1, 29)].mean(axis=1),
        'month': calendar_days.loc[day_idx, 'month'].to_numpy(dtype=np.int8),
        'weekday': calendar_days.loc[day_idx, 'wday'].to_numpy(dtype=np.int8),
        'event': calendar_days.loc[day_idx, 'event_name_1'].notna().to_numpy(dtype=np.int8),
    })
    for column in category_columns:
        X[column] = category_codes[column][series_idx]
    return X

rng = np.random.default_rng(42)
N_SAMPLES = 300_000
series_idx = rng.integers(0, len(sales), size=N_SAMPLES)
day_idx = rng.integers(365, TRAIN_END, size=N_SAMPLES)
X_train = make_features(series_idx, day_idx, train)
y_train = train[series_idx, day_idx]
display(X_train.head())


,lag_1,lag_7,lag_14,lag_28,lag_56,rolling_7,rolling_28,month,weekday,event,item_id,dept_id,cat_id,store_id,state_id
0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,8,5,0,1109,2,0,0,0
1,5.0,4.0,1.0,1.0,1.0,2.714286,1.535714,9,4,0,642,2,0,7,2
2,0.0,0.0,0.0,0.0,0.0,0.000000,0.071429,4,7,0,51,0,0,6,1
3,0.0,1.0,0.0,0.0,1.0,0.285714,0.428571,4,6,0,2622,6,2,4,1
4,0.0,0.0,1.0,0.0,1.0,0.000000,0.285714,3,7,0,2443,5,2,4,1


## 4. Train the global model

In [5]:
model = lgb.LGBMRegressor(
    objective='poisson', n_estimators=350, learning_rate=.05, num_leaves=64,
    min_child_samples=100, subsample=.8, colsample_bytree=.9,
    reg_lambda=.2, random_state=42, n_jobs=-1, verbosity=-1,
)
model.fit(X_train, y_train, categorical_feature=category_columns)


,num_leaves,64
,learning_rate,0.05
,n_estimators,350
,objective,'poisson'
,min_child_samples,100
,subsample,0.8
,colsample_bytree,0.9
,reg_lambda,0.2
,random_state,42
,n_jobs,-1
,verbosity,-1


## 5. Generate the 28-day recursive forecast

In [6]:
history = np.concatenate([train, np.zeros((len(sales), HORIZON), dtype=np.float32)], axis=1)
model_forecast = np.zeros_like(actual)
all_series = np.arange(len(sales))
for step in range(HORIZON):
    day = TRAIN_END + step
    X_future = make_features(all_series, np.full(len(sales), day), history)
    model_forecast[:, step] = np.maximum(0, model.predict(X_future)).astype(np.float32)
    history[:, day] = model_forecast[:, step]


In [7]:
def score(actual, predicted, history):
    error = actual - predicted
    scale = np.mean(np.diff(history, axis=1) ** 2, axis=1)
    usable = scale > 0
    denominator = np.abs(actual).sum()
    return {
        'MAE': float(np.abs(error).mean()),
        'WAPE': float(np.abs(error).sum() / denominator),
        'RMSSE': float(np.sqrt(np.mean(error[usable] ** 2, axis=1) / scale[usable]).mean()),
        'Bias': float(error.sum() / denominator),
    }


## 6. Compare LightGBM, the baseline and transparent hybrids

In [8]:
baseline = np.repeat(train[:, -28:].mean(axis=1, keepdims=True), HORIZON, axis=1)
candidates = {
    'Mean of last 28 days': baseline,
    '25% LightGBM hybrid': .75 * baseline + .25 * model_forecast,
    '50% LightGBM hybrid': .50 * baseline + .50 * model_forecast,
    '75% LightGBM hybrid': .25 * baseline + .75 * model_forecast,
    'Global LightGBM': model_forecast,
}
metrics = pd.DataFrame([{'model': n, **score(actual, p, train)} for n, p in candidates.items()]).sort_values('WAPE')
display(metrics.style.format({'MAE': '{:.4f}', 'WAPE': '{:.2%}', 'RMSSE': '{:.4f}', 'Bias': '{:.2%}'}))
px.bar(metrics.sort_values('WAPE', ascending=False), x='WAPE', y='model', orientation='h', text_auto='.1%', title='28-day holdout WAPE').show()


,model,MAE,WAPE,RMSSE,Bias
0,Mean of last 28 days,1.0657,73.86%,0.9240,3.91%
1,25% LightGBM hybrid,1.0681,74.03%,0.9214,1.57%
2,50% LightGBM hybrid,1.0817,74.97%,0.9276,-0.76%
3,75% LightGBM hybrid,1.1056,76.63%,0.9416,-3.10%
4,Global LightGBM,1.1388,78.93%,0.9625,-5.43%


## 7. Explain the model

In [9]:
importance = pd.DataFrame({'feature': model.feature_name_, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
px.bar(importance.sort_values('importance'), x='importance', y='feature', orientation='h', title='LightGBM feature importance').show()

best_name = metrics.iloc[0].model
best = candidates[best_name]
forecast_daily = pd.DataFrame({
    'date': calendar_days.loc[TRAIN_END:, 'date'].to_numpy(),
    'actual': actual.sum(axis=0),
    'forecast': best.sum(axis=0),
})
px.line(forecast_daily, x='date', y=['actual', 'forecast'], markers=True, title=f'Actual versus {best_name}').show()


## Conclusion
A complex model earns deployment only when it improves the fixed holdout. Keeping the pure LightGBM result visible prevents cherry-picking and makes the hybrid trade-off auditable.